## HMM Assignment

In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import csv
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal
from scipy.fft import fft, fftfreq
from hmmlearn import hmm
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

In [ ]:
NOTEBOOK_DIR = Path(os.getcwd())
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR

BASE_PATH = PROJECT_ROOT / "Recordings"
PROCESSED_PATH = PROJECT_ROOT / "data_processed"
REPORT_PATH = PROJECT_ROOT / "report"

for p in [PROCESSED_PATH, REPORT_PATH]:
    p.mkdir(exist_ok=True)

print(f"Project Root: {PROJECT_ROOT}")
print(f"Found {len(list(BASE_PATH.glob('**/*.csv')))} CSV files")

example = next(BASE_PATH.glob("Walking/*/Accelerometer.csv"), None)
print(f"Example file: {example}")

In [ ]:
walking_dirs = sorted((BASE_PATH / "Walking").glob("*"))
example_dir = walking_dirs[0]
print(f"Selected session: {example_dir.name}")
print(f"Path: {example_dir}")

acc_file = example_dir / "Accelerometer.csv"
gyro_file = example_dir / "Gyroscope.csv"

# Load CSVs
df_acc = pd.read_csv(acc_file)
df_gyro = pd.read_csv(gyro_file)

print(f"Accelerometer: {df_acc.shape[0]} rows")
print(f"Gyroscope:    {df_gyro.shape[0]} rows")

In [ ]:
print("\nAccelerometer sample:")
display(df_acc.head(3))
print("\nGyroscope sample:")
display(df_gyro.head(3))

In [ ]:
df_acc = pd.read_csv(acc_file)
df_gyro = pd.read_csv(gyro_file)

df_acc['seconds_elapsed'] = df_acc['seconds_elapsed'].round(6)
df_gyro['seconds_elapsed'] = df_gyro['seconds_elapsed'].round(6)

df_acc_sorted = df_acc.sort_values('seconds_elapsed')
df_gyro_sorted = df_gyro.sort_values('seconds_elapsed')

df_merged = pd.merge_asof(
    df_acc_sorted,
    df_gyro_sorted,
    on='seconds_elapsed',
    direction='nearest',
    tolerance=0.01
).dropna()

df_merged = df_merged.rename(columns={
    'x_x': 'acc_x', 'y_x': 'acc_y', 'z_x': 'acc_z',
    'x_y': 'gyro_x', 'y_y': 'gyro_y', 'z_y': 'gyro_z'
})

df_merged = df_merged.drop(columns=['time_x', 'time_y'], errors='ignore')

print("Merged columns:", df_merged.columns.tolist())
display(df_merged.head(3))

In [ ]:
time_diffs = np.diff(df_merged['seconds_elapsed'])
sampling_rate = 1.0 / np.median(time_diffs)
duration = df_merged['seconds_elapsed'].iloc[-1] - df_merged['seconds_elapsed'].iloc[0]

print(f"\nSAMPLING RATE: {sampling_rate:.2f} Hz")
print(f"DURATION: {duration:.1f} seconds")
print(f"TOTAL SAMPLES: {len(df_merged)}")

In [ ]:
sns.set_style("whitegrid")
fig, axes = plt.subplots(3, 2, figsize=(14, 10), sharex=True)
t = df_merged['seconds_elapsed']

acc_axes = ['acc_x', 'acc_y', 'acc_z']
acc_labels = ['X (forward)', 'Y (lateral)', 'Z (vertical)']

for i, (col, label) in enumerate(zip(acc_axes, acc_labels)):
    axes[i, 0].plot(t, df_merged[col], label=f'{label}', linewidth=1, color='tab:blue')
    axes[i, 0].set_ylabel(f'Acc (m/s²)')
    axes[i, 0].set_title(f'Accelerometer {label}')
    axes[i, 0].grid(True)

gyro_axes = ['gyro_x', 'gyro_y', 'gyro_z']
gyro_labels = ['X (roll)', 'Y (pitch)', 'Z (yaw)']

for i, (col, label) in enumerate(zip(gyro_axes, gyro_labels)):
    axes[i, 1].plot(t, df_merged[col], label=f'{label}', linewidth=1, color='tab:orange')
    axes[i, 1].set_ylabel(f'Gyro (rad/s)')
    axes[i, 1].set_title(f'Gyroscope {label}')
    axes[i, 1].grid(True)

axes[-1, 0].set_xlabel('Time (seconds)')
axes[-1, 1].set_xlabel('Time (seconds)')

activity_name = example_dir.parent.name 
plt.suptitle(f"Raw Sensor Data: {activity_name} Session\n"
             f"Sampling Rate: {sampling_rate:.1f} Hz | Duration: {duration:.0f}s", 
             fontsize=14, y=0.98)

plt.tight_layout()

plot_path = REPORT_PATH / f"fig_raw_signal_{activity_name.lower()}.png"
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
print(f"Plot saved: {plot_path}")
plt.show()

In [ ]:
REPORT_PATH.mkdir(exist_ok=True)

metadata_file = PROJECT_ROOT / "metadata.csv"

if not metadata_file.exists():
    with open(metadata_file, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['session_path', 'activity', 'phone_model', 
                         'sampling_rate_hz', 'duration_sec', 'num_samples'])

saved_plots = {}

activities = ['Still', 'Standing', 'Walking', 'Jumping']
print(f"Processing {len(activities)} activities...")

for activity in activities:
    activity_path = BASE_PATH / activity
    if not activity_path.exists():
        print(f"Warning: {activity_path} not found")
        continue
    
    session_dirs = sorted(activity_path.glob("*"))
    print(f"\n{activity}: {len(session_dirs)} sessions")
    
    for session_dir in tqdm(session_dirs, desc=activity):
        acc_file = session_dir / "Accelerometer.csv"
        gyro_file = session_dir / "Gyroscope.csv"
        
        if not (acc_file.exists() and gyro_file.exists()):
            print(f"  Missing files in {session_dir.name}")
            continue
        
        df_acc = pd.read_csv(acc_file)
        df_gyro = pd.read_csv(gyro_file)
        
        df_acc['seconds_elapsed'] = df_acc['seconds_elapsed'].round(6)
        df_gyro['seconds_elapsed'] = df_gyro['seconds_elapsed'].round(6)
        
        df_merged = pd.merge_asof(
            df_acc.sort_values('seconds_elapsed'),
            df_gyro.sort_values('seconds_elapsed'),
            on='seconds_elapsed',
            direction='nearest',
            tolerance=0.01
        ).dropna()
        
        if len(df_merged) == 0:
            print(f"  No overlap in {session_dir.name}")
            continue
        
        df_merged = df_merged.rename(columns={
            'x_x': 'acc_x', 'y_x': 'acc_y', 'z_x': 'acc_z',
            'x_y': 'gyro_x', 'y_y': 'gyro_y', 'z_y': 'gyro_z'
        })
        df_merged = df_merged.drop(columns=['time_x', 'time_y'], errors='ignore')
        
        time_diffs = np.diff(df_merged['seconds_elapsed'])
        if len(time_diffs) == 0:
            continue
        sampling_rate = 1.0 / np.median(time_diffs)
        duration = df_merged['seconds_elapsed'].iloc[-1] - df_merged['seconds_elapsed'].iloc[0]
        
        row = [
            str(session_dir.relative_to(PROJECT_ROOT)),
            activity,
            "Samsung A23",
            round(sampling_rate, 2),
            round(duration, 1),
            len(df_merged)
        ]
        with open(metadata_file, 'a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(row)
        
        if activity not in saved_plots:
            fig, axes = plt.subplots(3, 2, figsize=(14, 10), sharex=True)
            t = df_merged['seconds_elapsed']
            
            for i, col in enumerate(['acc_x', 'acc_y', 'acc_z']):
                axes[i, 0].plot(t, df_merged[col], linewidth=1, color='tab:blue')
                axes[i, 0].set_ylabel(f'Acc (m/s²)')
                axes[i, 0].set_title(f'Acc {["X","Y","Z"][i]}')
                axes[i, 0].grid(True)
            
            for i, col in enumerate(['gyro_x', 'gyro_y', 'gyro_z']):
                axes[i, 1].plot(t, df_merged[col], linewidth=1, color='tab:orange')
                axes[i, 1].set_ylabel(f'Gyro (rad/s)')
                axes[i, 1].set_title(f'Gyro {["X","Y","Z"][i]}')
                axes[i, 1].grid(True)
            
            axes[-1,0].set_xlabel('Time (s)')
            axes[-1,1].set_xlabel('Time (s)')
            plt.suptitle(f"{activity} – {sampling_rate:.1f} Hz, {duration:.0f}s", fontsize=14)
            plt.tight_layout()
            
            plot_path = REPORT_PATH / f"fig_raw_{activity.lower()}.png"
            plt.savefig(plot_path, dpi=150, bbox_inches='tight')
            plt.close()
            saved_plots[activity] = str(plot_path)
            print(f"  Plot saved: {plot_path.name}")

print("\nAll done! Metadata built.")

In [ ]:
meta = pd.read_csv(metadata_file)
summary = meta.groupby('activity').agg(
    num_sessions=('session_path', 'count'),
    total_duration=('duration_sec', 'sum'),
    avg_rate=('sampling_rate_hz', 'mean'),
    total_samples=('num_samples', 'sum')
).round(1)

summary['total_duration_min'] = (summary['total_duration'] / 60).round(1)
summary = summary[['num_sessions', 'total_duration_min', 'avg_rate', 'total_samples']]
print("\nDATA COLLECTION SUMMARY")
display(summary)

# Save for report
summary.to_csv(PROJECT_ROOT / "summary_table.csv")
print("Summary saved: summary_table.csv")

In [ ]:
WINDOW_SEC = 2.0          
OVERLAP = 0.5            
TARGET_RATE = 50.0       


FEATURE_NAMES = [
    'acc_mean_x', 'acc_mean_y', 'acc_mean_z',
    'acc_rms_x',  'acc_rms_y',  'acc_rms_z',
    'gyro_mean_x','gyro_mean_y','gyro_mean_z',
    'acc_sma',    'gyro_sma',
    'acc_gravity', 'tilt_pitch',           
    'acc_fft_peak','gyro_fft_peak',
    'acc_spectral_energy','gyro_spectral_energy'
]

print(f"Window: {WINDOW_SEC}s, Overlap: {OVERLAP*100}%, Features: {len(FEATURE_NAMES)}")

In [ ]:
def extract_features(window_df, rate):
    features = {}
    
    acc = window_df[['acc_x', 'acc_y', 'acc_z']].values
    gyro = window_df[['gyro_x', 'gyro_y', 'gyro_z']].values
    
    features['acc_mean_x'] = np.mean(acc[:,0])
    features['acc_mean_y'] = np.mean(acc[:,1])
    features['acc_mean_z'] = np.mean(acc[:,2])
    features['acc_rms_x'] = np.sqrt(np.mean(acc[:,0]**2))
    features['acc_rms_y'] = np.sqrt(np.mean(acc[:,1]**2))
    features['acc_rms_z'] = np.sqrt(np.mean(acc[:,2]**2))
    features['gyro_mean_x'] = np.mean(gyro[:,0])
    features['gyro_mean_y'] = np.mean(gyro[:,1])
    features['gyro_mean_z'] = np.mean(gyro[:,2])
    
    features['acc_sma'] = np.sum(np.abs(acc)) / len(acc)
    features['gyro_sma'] = np.sum(np.abs(gyro)) / len(gyro)
    
    acc_magnitude = np.linalg.norm(acc, axis=1)
    features['acc_gravity'] = np.mean(acc_magnitude)  

    features['tilt_pitch'] = np.arctan2(np.mean(acc[:, 1]), np.mean(acc[:, 2])) 

    N = len(acc)
    freq = fftfreq(N, 1/rate)
    mask = (freq >= 0) & (freq <= 10) 
    
    acc_fft = np.abs(fft(acc, axis=0))
    gyro_fft = np.abs(fft(gyro, axis=0))
    
    features['acc_fft_peak'] = np.max(acc_fft[mask], axis=0).sum()
    features['gyro_fft_peak'] = np.max(gyro_fft[mask], axis=0).sum()
    
    features['acc_spectral_energy'] = np.sum(acc_fft[mask]**2)
    features['gyro_spectral_energy'] = np.sum(gyro_fft[mask]**2)
    
    return [features[name] for name in FEATURE_NAMES]

In [ ]:
TRIM_SEC = 2.0 

X_seqs = []
y_seqs = []
lengths = []
session_ids = []

meta = pd.read_csv(metadata_file)

for _, row in tqdm(meta.iterrows(), total=len(meta), desc="Windowing + Trim"):
    session_path = PROJECT_ROOT / row['session_path']
    acc_file = session_path / "Accelerometer.csv"
    gyro_file = session_path / "Gyroscope.csv"
    
    if not (acc_file.exists() and gyro_file.exists()):
        continue
    
    df_acc = pd.read_csv(acc_file)
    df_gyro = pd.read_csv(gyro_file)
    df_acc['seconds_elapsed'] = df_acc['seconds_elapsed'].round(6)
    df_gyro['seconds_elapsed'] = df_gyro['seconds_elapsed'].round(6)
    
    df_merged = pd.merge_asof(
        df_acc.sort_values('seconds_elapsed'),
        df_gyro.sort_values('seconds_elapsed'),
        on='seconds_elapsed',
        direction='nearest',
        tolerance=0.01
    ).dropna()
    
    df_merged = df_merged.rename(columns={
        'x_x': 'acc_x', 'y_x': 'acc_y', 'z_x': 'acc_z',
        'x_y': 'gyro_x', 'y_y': 'gyro_y', 'z_y': 'gyro_z'
    }).drop(columns=['time_x', 'time_y'], errors='ignore')
    
    if len(df_merged) < 50:
        continue
    
    rate = row['sampling_rate_hz']
    trim_samples = int(TRIM_SEC * rate)
    
    if len(df_merged) > 2 * trim_samples:
        df_trimmed = df_merged.iloc[trim_samples:-trim_samples].copy()
    else:
        print(f"  Skipping {session_path.name}: too short after trim")
        continue
    
    if len(df_trimmed) < 50:
        continue
    
    duration_after_trim = df_trimmed['seconds_elapsed'].iloc[-1] - df_trimmed['seconds_elapsed'].iloc[0]
    print(f"  {row['activity']} | {session_path.name} | {duration_after_trim:.1f}s after trim")
    
    window_samples = int(WINDOW_SEC * rate)
    step_samples = int(window_samples * (1 - OVERLAP))
    
    features_seq = []
    labels_seq = []
    
    for start in range(0, len(df_trimmed) - window_samples + 1, step_samples):
        window = df_trimmed.iloc[start:start + window_samples]
        if len(window) != window_samples:
            continue
        feats = extract_features(window, rate)
        features_seq.append(feats)
        labels_seq.append(row['activity'])
    
    if len(features_seq) > 0:
        X_seqs.append(np.array(features_seq))
        y_seqs.append(np.array(labels_seq))
        lengths.append(len(features_seq))
        session_ids.append(row['session_path'])

print(f"Extracted {sum(lengths)} windows from {len(X_seqs)} sessions (after trimming)")